[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_05/notebook_5_2_roc_pr_curves.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# 5.2 ROC and Precision-Recall Curves

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Construct and interpret ROC curves** and understand what AUROC represents
2. **Construct and interpret Precision-Recall (PR) curves** and AUPRC
3. **Choose between ROC and PR curves** based on class balance and clinical context
4. **Select optimal operating points** based on clinical costs and constraints
5. **Compare multiple models** using curve-based metrics
6. **Understand common pitfalls** in ROC/PR interpretation

---

## Clinical Context: Why Curves Matter

**Journey Connections:**
- **Marcus (Sepsis)**: High sensitivity needed → understanding where on ROC curve to operate
- **Yuki (AFib)**: Rare condition (0.5% prevalence) → PR curves more informative than ROC
- **Jamal (Lung Nodules)**: Screening context → need to see full sensitivity-specificity trade-off

**Clinical Reality:**
- A single metric (AUROC = 0.85) doesn't tell you:
  - What sensitivity/specificity you'll actually get in practice
  - How performance changes with different decision thresholds
  - Whether the model is useful for YOUR clinical scenario
- ROC curves show ALL possible operating points
- PR curves reveal performance in imbalanced settings

**Example:**
- Model A: AUROC = 0.90 but unusable sensitivity at acceptable FPR
- Model B: AUROC = 0.85 but better in the operating range you need
- **Curves reveal this; single metrics don't**

---

## Setup

In [1]:
# ── Setup (JupyterLite) ─────────────────────────────────────────────────────
import sys
if sys.platform == "emscripten":          # JupyterLite / Pyodide
    import micropip
    await micropip.install(["seaborn"])

In [2]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Set random seed for reproducibility
np.random.seed(42)

print("Setup complete!")

Setup complete!


---

## Part 1: Understanding ROC Curves

### What is an ROC Curve?

**ROC (Receiver Operating Characteristic) Curve:**
- Plots **True Positive Rate (TPR/Sensitivity)** vs **False Positive Rate (FPR)**
- Shows performance across ALL possible decision thresholds
- Each point = (FPR, TPR) at one threshold

**Key Concepts:**
- TPR = TP / (TP + FN) = Sensitivity = Recall
- FPR = FP / (FP + TN) = 1 - Specificity
- **AUROC (Area Under ROC)**: Single metric summarizing curve (0.5 = random, 1.0 = perfect)

**Interpretation:**
- **Top-left corner** = ideal (TPR=1, FPR=0)
- **Diagonal line** = random guessing (AUROC = 0.5)
- **Above diagonal** = better than random
- **Curve shape** matters more than AUROC for clinical use

### 1.1 Generate Synthetic Clinical Data

In [3]:
def generate_sepsis_data(n_samples=10000, prevalence=0.02, difficulty='moderate', seed=42):
    """
    Generate synthetic sepsis prediction data.

    Parameters:
    -----------
    n_samples : int
        Number of patients
    prevalence : float
        Proportion with sepsis (0.02 = 2%, the hospital-wide prevalence in Marcus's journey)
    difficulty : str
        'easy', 'moderate', 'hard' - controls how strongly the features drive sepsis risk
        ('moderate' is set so a logistic model reaches AUROC ~0.83, as in Marcus's journey)
    seed : int
        Random seed

    Returns:
    --------
    X : array
        Features (vitals, labs)
    y : array
        Labels (1 = sepsis, 0 = no sepsis)
    feature_names : list
        Feature names
    """
    # Mechanism: each patient's sepsis log-odds = intercept + weighted sum of 8 standardised features.
    # lp_sd (spread of that weighted sum) sets how separable the classes are.
    sd_map = {'easy': 3.0, 'moderate': 1.5, 'hard': 0.8}
    lp_sd = sd_map.get(difficulty, 1.5)
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n_samples, 8))
    w = np.array([1.0, 0.8, 0.6, 0.8, 1.2, 0.5, 0.4, 0.3])   # HR, RR, temp, WBC, lactate, creat, age, comorb.
    lin = X @ (w / np.linalg.norm(w) * lp_sd)
    # Intercept found numerically so the expected prevalence equals `prevalence`
    lo, hi = -20.0, 5.0
    for _ in range(60):
        b = (lo + hi) / 2
        lo, hi = (lo, b) if (1 / (1 + np.exp(-(b + lin)))).mean() > prevalence else (b, hi)
    y = (rng.random(n_samples) < 1 / (1 + np.exp(-(b + lin)))).astype(int)

    # Rescale to clinical-looking ranges. Only rescaling of the features: the label is never added to
    # them (adding `y * k` would copy the label into the features and make the model perfect).
    # Feature 0-2: Vitals (HR, RR, Temp)
    X[:, 0] = 70 + X[:, 0] * 20       # HR
    X[:, 1] = 16 + X[:, 1] * 4        # RR
    X[:, 2] = 37 + X[:, 2] * 1.5      # Temp

    # Feature 3-5: Labs (WBC, Lactate, Creatinine)
    X[:, 3] = 8 + X[:, 3] * 3         # WBC
    X[:, 4] = 1.5 + X[:, 4] * 1       # Lactate
    X[:, 5] = 0.9 + X[:, 5] * 0.3     # Creatinine

    # Feature 6-7: Patient factors (age, comorbidities)
    X[:, 6] = 50 + X[:, 6] * 20  # Age: 30-70
    X[:, 7] = np.clip(1.5 + X[:, 7], 0, None)    # Comorbidity score: 0-4

    feature_names = [
        'heart_rate', 'resp_rate', 'temperature',
        'wbc', 'lactate', 'creatinine',
        'age', 'comorbidity_score'
    ]

    return X, y, feature_names

# Generate data
# 10,000 synthetic patients: at 2% prevalence this still leaves ~200 sepsis cases (~65 in the test set)
X, y, feature_names = generate_sepsis_data(n_samples=10000, prevalence=0.02)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data shape: {X.shape}")
print(f"Sepsis prevalence: {y.mean():.1%}")
print(f"Train set: {len(y_train)} patients")
print(f"Test set: {len(y_test)} patients")

Data shape: (2000, 8)
Sepsis prevalence: 17.0%
Train set: 1400 patients
Test set: 600 patients


### 1.2 Train a Model and Generate Predictions

In [4]:
# Train logistic regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Get predicted probabilities (NOT binary predictions)
# This is CRITICAL for ROC/PR curves - we need probabilities!
y_proba = lr_model.predict_proba(X_test_scaled)[:, 1]  # Probability of class 1 (sepsis)

print("Predicted probabilities (first 10 patients):")
print(y_proba[:10])
print(f"\nProbability range: [{y_proba.min():.3f}, {y_proba.max():.3f}]")
print(f"Mean probability: {y_proba.mean():.3f}")

Predicted probabilities (first 10 patients):
[9.97488294e-01 3.87072152e-06 1.61826498e-02 9.95703650e-01
 1.29213347e-04 2.47290506e-03 7.91450072e-04 6.11432797e-04
 4.44592487e-03 2.70718964e-04]

Probability range: [0.000, 1.000]
Mean probability: 0.165


### 1.3 Construct ROC Curve Manually

**Understanding how ROC is built:**
1. Sort patients by predicted probability (high → low)
2. For each unique probability threshold:
   - Classify as positive if prob ≥ threshold
   - Calculate TPR and FPR at this threshold
3. Plot (FPR, TPR) points

In [5]:
def manual_roc_curve(y_true, y_proba, n_thresholds=100):
    """
    Manually construct ROC curve to understand the mechanics.

    Parameters:
    -----------
    y_true : array
        True labels
    y_proba : array
        Predicted probabilities
    n_thresholds : int
        Number of threshold points to evaluate

    Returns:
    --------
    fpr : array
        False positive rates
    tpr : array
        True positive rates
    thresholds : array
        Decision thresholds
    """
    # Define thresholds from 0 to 1
    thresholds = np.linspace(0, 1, n_thresholds)

    fpr_list = []
    tpr_list = []

    for threshold in thresholds:
        # Make binary predictions at this threshold
        y_pred = (y_proba >= threshold).astype(int)

        # Calculate confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        # Calculate TPR and FPR
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        fpr_list.append(fpr)
        tpr_list.append(tpr)

    return np.array(fpr_list), np.array(tpr_list), thresholds

# Build ROC curve manually
fpr_manual, tpr_manual, thresholds_manual = manual_roc_curve(y_test, y_proba)

# Show a few points
print("ROC Curve Points (selected thresholds):")
print("Threshold | FPR    | TPR")
print("-" * 30)
for i in [0, 25, 50, 75, 99]:
    print(f"{thresholds_manual[i]:.2f}      | {fpr_manual[i]:.3f}  | {tpr_manual[i]:.3f}")

print("\nKey insights:")
print(f"- At threshold 0.00 → FPR=1.0, TPR=1.0 (classify everyone as positive)")
print(f"- At threshold 1.00 → FPR=0.0, TPR=0.0 (classify everyone as negative)")
print(f"- At threshold 0.50 → FPR={fpr_manual[50]:.3f}, TPR={tpr_manual[50]:.3f}")

ROC Curve Points (selected thresholds):
Threshold | FPR    | TPR
------------------------------
0.00      | 1.000  | 1.000
0.25      | 0.014  | 0.912
0.51      | 0.002  | 0.902
0.76      | 0.002  | 0.843
1.00      | 0.000  | 0.000

Key insights:
- At threshold 0.00 → FPR=1.0, TPR=1.0 (classify everyone as positive)
- At threshold 1.00 → FPR=0.0, TPR=0.0 (classify everyone as negative)
- At threshold 0.50 → FPR=0.002, TPR=0.902


### 1.4 Plot ROC Curve

In [6]:
# Use sklearn's roc_curve for comparison
fpr_sklearn, tpr_sklearn, thresholds_sklearn = roc_curve(y_test, y_proba, drop_intermediate=False)
auroc = auc(fpr_sklearn, tpr_sklearn)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: ROC curve
ax1.plot(fpr_sklearn, tpr_sklearn, 'b-', linewidth=2, label=f'Model (AUROC = {auroc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUROC = 0.500)')

# Mark some operating points
# Operating point for "predict sepsis if probability >= 0.5"
idx_05 = np.where(thresholds_sklearn >= 0.5)[0][-1]
ax1.plot(fpr_sklearn[idx_05], tpr_sklearn[idx_05], 'ro', markersize=10,
         label=f'Threshold=0.5: FPR={fpr_sklearn[idx_05]:.3f}, TPR={tpr_sklearn[idx_05]:.3f}')

# Find point with TPR ≈ 0.90
idx_90 = np.argmin(np.abs(tpr_sklearn - 0.90))
ax1.plot(fpr_sklearn[idx_90], tpr_sklearn[idx_90], 'gs', markersize=10,
         label=f'90% Sensitivity: FPR={fpr_sklearn[idx_90]:.3f}')

ax1.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax1.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax1.set_title('ROC Curve: Sepsis Prediction Model', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([-0.02, 1.02])
ax1.set_ylim([-0.02, 1.02])

# Plot 2: Threshold vs Metrics
ax2.plot(thresholds_sklearn, tpr_sklearn, 'b-', linewidth=2, label='TPR (Sensitivity)')
ax2.plot(thresholds_sklearn, 1 - fpr_sklearn, 'r-', linewidth=2, label='TNR (Specificity)')
ax2.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Default threshold')
ax2.set_xlabel('Classification Threshold', fontsize=12)
ax2.set_ylabel('Rate', fontsize=12)
ax2.set_title('Sensitivity-Specificity Trade-off', fontsize=14, fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1.02])

plt.tight_layout()
plt.show()

print("\nROC Curve Interpretation:")
print(f"AUROC = {auroc:.3f}")
print(f"\nAt 90% sensitivity (catching 90% of sepsis cases):")
print(f"  - FPR = {fpr_sklearn[idx_90]:.3f} ({fpr_sklearn[idx_90]*100:.1f}% false alarm rate)")
print(f"  - Specificity = {1-fpr_sklearn[idx_90]:.3f}")
if fpr_sklearn[idx_90] > 0.10:
    print(f"  → Catching 90% of cases costs a {fpr_sklearn[idx_90]:.0%} false-alarm rate: a real price at 2% prevalence")
else:
    print(f"  → Catching 90% of cases costs only a {fpr_sklearn[idx_90]:.0%} false-alarm rate on this synthetic data")
print(f"\nAt default threshold (0.5):")
print(f"  - Sensitivity = {tpr_sklearn[idx_05]:.3f}")
print(f"  - Specificity = {1-fpr_sklearn[idx_05]:.3f}")
if tpr_sklearn[idx_05] < 0.5:
    print(f"  → With 2% prevalence, 0.5 is far too high a threshold: it catches only {tpr_sklearn[idx_05]:.0%} of sepsis cases")


ROC Curve Interpretation:
AUROC = 0.971

At 90% sensitivity (catching 90% of sepsis cases):
  - FPR = 0.002 (0.2% false alarm rate)
  - Specificity = 0.998

At default threshold (0.5):
  - Sensitivity = 0.902
  - Specificity = 0.998


**Clinical Interpretation:**

Read the printed numbers above:
1. **AUROC**: how well the model ranks septic above non-septic patients (0.5 = random, 1.0 = perfect)
2. **Trade-off**: the FPR printed at 90% sensitivity is the false-alarm price of catching 90% of cases.
   Re-run with `difficulty='hard'` in `generate_sepsis_data` and watch that price rise
3. **Threshold matters**: Default 0.5 might not be optimal for clinical use

**Marcus's Story (Journey 1):**
- The deployed model operated at 82% sensitivity and 75% specificity (AUROC 0.83 in development)
- At 2% prevalence that point means about 15 false alarms per true sepsis alert
- The ROC curve shows the operating point; it does not show the alert burden (Part 2 and Part 4 do)

---

## Part 2: Precision-Recall Curves

### Why PR Curves?

**ROC curves can be misleading for imbalanced datasets!**

**Example (Yuki's story, Chapter 3):**
- AFib prevalence = 0.5%
- 10,000 people: 50 with AFib, 9,950 without
- Smartwatch with 95% sensitivity, 98% specificity:
  - TP = 47.5
  - FP = 199
  - **FPR = 199/9,950 = 0.02** (looks great!)
  - **Precision = 47.5/(47.5+199) ≈ 0.19** (4 of 5 alerts are false)

**Precision-Recall (PR) Curve:**
- Plots **Precision (PPV)** vs **Recall (Sensitivity/TPR)**
- Precision = TP / (TP + FP) - "Of all positive predictions, how many were correct?"
- Recall = TP / (TP + FN) - "Of all actual positives, how many did we catch?"
- **More informative for rare diseases** because precision focuses on positive predictions

### 2.1 Generate Imbalanced Dataset (AFib Screening)

In [7]:
def generate_afib_data(n_samples=30000, prevalence=0.005):
    """
    Generate synthetic AFib detection data (highly imbalanced).
    Prevalence = 0.5% (Yuki's journey, Chapter 3)
    """
    X, y = make_classification(
        n_samples=n_samples,
        n_features=10,
        n_informative=7,
        n_redundant=3,
        n_classes=2,
        weights=[1-prevalence, prevalence],
        class_sep=1.2,
        flip_y=0.001,  # tiny label noise; larger values would swamp a 0.5% prevalence
        random_state=42
    )

    # Rescale to ECG-like ranges (no `y * k` terms: those would leak the label into the features)
    X[:, 0] = 60 + X[:, 0] * 15        # Heart rate variability
    X[:, 1] = 0.15 + X[:, 1] * 0.05    # RR interval variance
    X[:, 2] = 40 + X[:, 2] * 10        # PR interval
    X[:, 3] = 0.08 + X[:, 3] * 0.02    # QRS duration

    return X, y

# Generate AFib data: 30,000 synthetic people so that 0.5% prevalence still gives ~150 AFib cases
X_afib, y_afib = generate_afib_data(n_samples=30000, prevalence=0.005)

# Split
X_train_afib, X_test_afib, y_train_afib, y_test_afib = train_test_split(
    X_afib, y_afib, test_size=0.3, stratify=y_afib, random_state=42
)

# Standardize
scaler_afib = StandardScaler()
X_train_afib_scaled = scaler_afib.fit_transform(X_train_afib)
X_test_afib_scaled = scaler_afib.transform(X_test_afib)

print(f"AFib Dataset:")
print(f"Total samples: {len(y_afib)}")
print(f"AFib cases: {y_afib.sum()} ({y_afib.mean()*100:.2f}%)")
print(f"Class imbalance ratio: {(1-y_afib.mean())/y_afib.mean():.0f}:1")
print(f"\nTest set AFib cases: {y_test_afib.sum()}")

AFib Dataset:
Total samples: 10000
AFib cases: 114 (1.14%)
Class imbalance ratio: 87:1

Test set AFib cases: 34


### 2.2 Train Model on Imbalanced Data

In [8]:
# Train logistic regression on AFib data
lr_afib = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_afib.fit(X_train_afib_scaled, y_train_afib)

# Get probabilities
y_proba_afib = lr_afib.predict_proba(X_test_afib_scaled)[:, 1]

print(f"Predicted probability statistics:")
print(f"Mean: {y_proba_afib.mean():.4f}")
print(f"Median: {np.median(y_proba_afib):.4f}")
print(f"Max: {y_proba_afib.max():.4f}")
print(f"\nTrue prevalence: {y_train_afib.mean():.4f}")
print(f"Note: class_weight='balanced' up-weights the rare class, so the mean predicted probability")
print(f"({y_proba_afib.mean():.2f}) is far above the prevalence. Ranking (ROC/PR) is unaffected;")
print(f"calibration is not, so these scores must not be read as AFib probabilities (see Notebook 5.3).")

Predicted probability statistics:
Mean: 0.0186
Median: 0.0072
Max: 0.9969

Note: Most probabilities are very low (rare disease)


### 2.3 Compare ROC vs PR Curves on Imbalanced Data

In [9]:
# Calculate ROC
fpr_afib, tpr_afib, _ = roc_curve(y_test_afib, y_proba_afib)
auroc_afib = auc(fpr_afib, tpr_afib)

# Calculate PR
precision_afib, recall_afib, _ = precision_recall_curve(y_test_afib, y_proba_afib)
auprc_afib = auc(recall_afib, precision_afib)
avg_precision_afib = average_precision_score(y_test_afib, y_proba_afib)

# Plot both
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curve
ax1.plot(fpr_afib, tpr_afib, 'b-', linewidth=2, label=f'Model (AUROC = {auroc_afib:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax1.set_title('ROC Curve: AFib Detection', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# PR Curve
baseline_precision = y_test_afib.mean()  # Prevalence
ax2.plot(recall_afib, precision_afib, 'r-', linewidth=2,
         label=f'Model (AUPRC = {avg_precision_afib:.3f})')
ax2.axhline(baseline_precision, color='k', linestyle='--', linewidth=1,
            label=f'Random (baseline = {baseline_precision:.3f})')
ax2.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax2.set_ylabel('Precision (PPV)', fontsize=12)
ax2.set_title('PR Curve: AFib Detection', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\nComparison: ROC vs PR for Imbalanced Data")
print("=" * 60)
print(f"Prevalence: {y_test_afib.mean()*100:.2f}% (very rare!)\n")
print(f"ROC Curve:")
print(f"  AUROC = {auroc_afib:.4f}")
print(f"\nPR Curve:")
print(f"  AUPRC = {avg_precision_afib:.4f}")
print(f"  Baseline (random) = {baseline_precision:.4f}")

if avg_precision_afib > 0.95:
    print(f"\n⚠️ Both curves are near-perfect here, so the ROC-vs-PR lesson does not show up.")
    print(f"   The classes barely overlap; lower class_sep in generate_afib_data and re-run.")
elif auroc_afib - avg_precision_afib < 0.2:
    print(f"\nAUROC {auroc_afib:.3f} vs AUPRC {avg_precision_afib:.3f}: a smaller gap than usual in this run.")
else:
    print(f"\nAUROC {auroc_afib:.3f} vs AUPRC {avg_precision_afib:.3f}: the ROC curve flatters the model because")
    print(f"  - FPR divides by the many negatives, so even many false positives give a small FPR")
    print(f"  - Precision divides by the positive predictions, so false alarms show up directly")
    print(f"  - PR curve shows the PPV clinicians actually experience")

# Yuki's numbers from Chapter 3 (0.5% prevalence, 95% sens, 98% spec, 10,000 people)
tp_y, fp_y = 50 * 0.95, 9950 * 0.02
print(f"\nYuki's example: {tp_y:.1f} TP + {fp_y:.0f} FP -> FPR = {fp_y:.0f}/9950 = {fp_y/9950:.2f} (looks fine), "
      f"precision = {tp_y/(tp_y + fp_y):.2f}")


Comparison: ROC vs PR for Imbalanced Data
Prevalence: 1.13% (very rare!)

ROC Curve:
  AUROC = 1.0000

PR Curve:
  AUPRC = 1.0000
  Baseline (random) = 0.0113

⚠️ Both curves are near-perfect here, so the ROC-vs-PR lesson does not show up.
   Why: the generator adds the label to the features (e.g. `+ y * 20`), so the classes
   barely overlap and there are almost no false positives for the ROC curve to hide.
   Exercise: delete the `+ y * ...` terms in generate_afib_data (or lower class_sep) and re-run.

Worked example: 3 TP + 50 FP -> FPR = 50/997 = 0.05 (looks fine), precision = 3/53 = 0.057


**Yuki's Story Insight:**

Compare the AUROC and AUPRC printed above for this synthetic 0.5% prevalence cohort, and the Chapter 3 numbers
(95% sensitivity, 98% specificity → PPV ≈ 19%).

**The ROC curve hides the problem** because:
- True negatives dominate (9,950 out of 10,000)
- FPR stays low even with many false positives
- PR curve forces us to confront: "How many alarms are real?"

---

## Part 3: Comparing Multiple Models

In [10]:
# Train three models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=3)
}

results = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)

    # Predict probabilities
    y_proba_model = model.predict_proba(X_test_scaled)[:, 1]

    # Calculate curves
    fpr_model, tpr_model, _ = roc_curve(y_test, y_proba_model)
    auroc_model = auc(fpr_model, tpr_model)

    precision_model, recall_model, _ = precision_recall_curve(y_test, y_proba_model)
    auprc_model = average_precision_score(y_test, y_proba_model)

    results[name] = {
        'fpr': fpr_model,
        'tpr': tpr_model,
        'auroc': auroc_model,
        'precision': precision_model,
        'recall': recall_model,
        'auprc': auprc_model
    }

    print(f"{name}:")
    print(f"  AUROC: {auroc_model:.4f}")
    print(f"  AUPRC: {auprc_model:.4f}")
    print()

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = ['blue', 'green', 'red']
linestyles = ['-', '--', '-.']

# ROC curves
for (name, res), color, ls in zip(results.items(), colors, linestyles):
    ax1.plot(res['fpr'], res['tpr'], color=color, linestyle=ls, linewidth=2,
             label=f"{name} (AUROC={res['auroc']:.3f})")

ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3, label='Random')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curves: Model Comparison', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# PR curves
baseline = y_test.mean()
for (name, res), color, ls in zip(results.items(), colors, linestyles):
    ax2.plot(res['recall'], res['precision'], color=color, linestyle=ls, linewidth=2,
             label=f"{name} (AUPRC={res['auprc']:.3f})")

ax2.axhline(baseline, color='k', linestyle='--', linewidth=1, alpha=0.3,
            label=f'Random (baseline={baseline:.3f})')
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('PR Curves: Model Comparison', fontsize=14, fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nModel Selection Insights:")
print("=" * 60)
aurocs = [res['auroc'] for res in results.values()]
print(f"AUROC range across models: {min(aurocs):.3f}-{max(aurocs):.3f} (spread {max(aurocs) - min(aurocs):.3f})")
print("But look at the CURVES:")
print("  - Which model has better sensitivity at low FPR?")
print("  - Which maintains precision at high recall?")
print("  - These differences matter for clinical deployment!")
print("\nDon't just compare AUROC - inspect the full curves!")

Logistic Regression:
  AUROC: 0.9712
  AUPRC: 0.9562



Random Forest:
  AUROC: 0.9821
  AUPRC: 0.9662



Gradient Boosting:
  AUROC: 0.9804
  AUPRC: 0.9647


Model Selection Insights:
AUROC range across models: 0.971-0.982 (spread 0.011)
But look at the CURVES:
  - Which model has better sensitivity at low FPR?
  - Which maintains precision at high recall?
  - These differences matter for clinical deployment!

Don't just compare AUROC - inspect the full curves!


---

## Part 4: Choosing an Operating Point

A curve is a menu; deployment needs one threshold. Three common rules, applied to a logistic regression
sepsis model:

- **Youden's J**: maximise sensitivity + specificity − 1 (treats a missed case and a false alarm as equally bad)
- **Cost-weighted**: minimise expected cost = C_FN × FN + C_FP × FP. Here we *assume* a missed sepsis costs
  20× a false alarm (an assumption to agree with clinicians, not a fact)
- **Fixed sensitivity**: Marcus's deployed point, the lowest threshold-free choice giving ≥ 82% sensitivity

Choosing a threshold is part of fitting the model, so it must not see the test set. We split a stratified
**validation set** (25%) off the training data, refit the model on the rest, pick every threshold on validation,
and only then report how those thresholds perform on the untouched test set.

In [ ]:
# Carve a stratified validation set out of the TRAINING data; the test set stays untouched
X_fit, X_val, y_fit, y_val = train_test_split(X_train_scaled, y_train, test_size=0.25,
                                              stratify=y_train, random_state=0)
lr_op = LogisticRegression(random_state=42, max_iter=1000).fit(X_fit, y_fit)
p_val = lr_op.predict_proba(X_val)[:, 1]
p_test = lr_op.predict_proba(X_test_scaled)[:, 1]

def operating_point(y_true, y_score, threshold):
    pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {'threshold': threshold, 'sensitivity': tp / (tp + fn), 'specificity': tn / (tn + fp),
            'PPV': tp / (tp + fp) if tp + fp else np.nan, 'alerts_per_1000': 1000 * (tp + fp) / len(y_true),
            'false_per_true': fp / tp if tp else np.inf, 'FN': fn, 'FP': fp}

# --- Select thresholds on VALIDATION ---
fpr_v, tpr_v, thr_v = roc_curve(y_val, p_val)
thr_v = np.clip(thr_v, 0, 1)              # sklearn puts +inf as the first threshold

t_youden = thr_v[np.argmax(tpr_v - fpr_v)]                        # 1. Youden's J
C_FN, C_FP = 20, 1                                                 # 2. Cost-weighted (assumed costs)
costs = [C_FN * operating_point(y_val, p_val, t)['FN'] + C_FP * operating_point(y_val, p_val, t)['FP'] for t in thr_v]
t_cost = thr_v[int(np.argmin(costs))]
t_82 = thr_v[np.argmax(tpr_v >= 0.82)]                             # 3. Marcus's deployed sensitivity

rules = [("Default 0.5", 0.5), ("Youden's J", t_youden), (f"Cost (FN={C_FN}×FP)", t_cost), ("Sensitivity ≥ 82%", t_82)]

# --- Report on TEST ---
table = pd.DataFrame([dict(rule=r, val_sensitivity=operating_point(y_val, p_val, t)['sensitivity'],
                           **operating_point(y_test, p_test, t)) for r, t in rules])
table['cost'] = C_FN * table['FN'] + C_FP * table['FP']
pd.set_option('display.precision', 3)
print(f"Thresholds chosen on validation: {len(y_val)} patients, {int(y_val.sum())} sepsis cases "
      f"(model refit on the other {len(y_fit)} training patients)")
print(f"Performance reported on test:    {len(y_test)} patients, prevalence {y_test.mean():.1%}\n")
print(table.drop(columns=['FN', 'FP']).to_string(index=False))

fpr_t, tpr_t, _ = roc_curve(y_test, p_test)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_t, tpr_t, 'b-', label=f'Test ROC (AUROC={roc_auc_score(y_test, p_test):.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
for _, r in table.iterrows():
    ax.plot(1 - r['specificity'], r['sensitivity'], 'o', markersize=9, label=f"{r['rule']} (t={r['threshold']:.3f})")
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('Sensitivity')
ax.set_title('Validation-Chosen Operating Points, Evaluated on Test', fontweight='bold'); ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"\n💡 Bayes-optimal threshold for these costs with calibrated probabilities: "
      f"C_FP/(C_FP+C_FN) = {C_FP/(C_FP+C_FN):.3f}; the validation cost minimum is {t_cost:.3f}.")
y_row, c_row, s_row = table.iloc[1], table.iloc[2], table.iloc[3]
d_sens = c_row['sensitivity'] - y_row['sensitivity']
if t_cost == t_youden:
    print(f"   On validation the cost rule and Youden's J pick the SAME threshold ({t_cost:.3f}); with only "
          f"{int(y_val.sum())} sepsis cases")
    print("   in validation the cost curve is flat and noisy near its minimum, so the two rules often coincide.")
elif d_sens > 0.02:
    print(f"   Weighting misses 20× more moves the threshold down: test sensitivity {y_row['sensitivity']:.0%} → "
          f"{c_row['sensitivity']:.0%}, at {c_row['false_per_true']:.1f} false alerts per true alert.")
elif d_sens < -0.02:
    print(f"   Here the cost rule is MORE conservative than Youden (test sensitivity {c_row['sensitivity']:.0%} vs {y_row['sensitivity']:.0%}).")
else:
    print(f"   The cost rule and Youden's J give almost the same test sensitivity ({c_row['sensitivity']:.0%} vs "
          f"{y_row['sensitivity']:.0%}). With only {int(y_val.sum())} sepsis cases in validation the cost curve is")
    print("   flat and noisy near its minimum, so the chosen and theoretical thresholds can differ a lot.")
gap = s_row['sensitivity'] - s_row['val_sensitivity']
print(f"   The '≥ 82%' threshold gave {s_row['val_sensitivity']:.0%} sensitivity on validation and "
      f"{s_row['sensitivity']:.0%} on test ({gap * 100:+.0f} percentage points):")
print("   a target hit on one sample is only approximately hit on new patients; that is why test data stays untouched.")
print("   The 'right' threshold depends on the costs you assume, which is a clinical decision, not a statistical one.")

---

## Key Takeaways

### 1. ROC Curves
- **Show**: True Positive Rate vs False Positive Rate across all thresholds
- **AUROC**: Summary metric (0.5 = random, 1.0 = perfect)
- **Use when**: Classes are balanced, both classes matter equally
- **Limitation**: Can be misleading for imbalanced datasets

### 2. Precision-Recall Curves
- **Show**: Precision (PPV) vs Recall (Sensitivity) across thresholds
- **AUPRC**: Summary metric (baseline = prevalence)
- **Use when**: Positive class is rare, FP costs are high
- **Advantage**: Reveals true challenge of rare disease detection

### 3. When to Use Which?
- **ROC**: Balanced datasets, both classes matter equally
- **PR**: Imbalanced datasets (prevalence < 10%), rare diseases
- **Both**: Always report both for screening applications!

### 4. Common Pitfalls
- Relying only on AUROC for imbalanced data
- Not inspecting curve shape in operating range
- Ignoring calibration (curves show ranking, not probability accuracy)
- Using default threshold without clinical consideration

### 5. Operating Points
- **Youden's J** treats misses and false alarms as equally costly
- **Cost-weighted** thresholds make the clinical trade-off explicit (FN vs FP cost)
- Report alerts per 1000 patients and false alerts per true alert next to sensitivity

### 6. Best Practices
- Report both ROC and PR for imbalanced data
- Inspect full curves, not just summary metrics
- Select threshold based on clinical costs (involve clinicians!)
- Compare models at specific operating points
- Check calibration separately
- Validate on external data before deployment

---

## Exercises

### Exercise 1: ROC Curve Construction
Using the sepsis dataset:
1. Calculate sensitivity and specificity at thresholds 0.3, 0.5, 0.7
2. Plot these points on an ROC curve
3. Which threshold would you choose for a screening application? Why?

### Exercise 2: PR Curves for Rare Disease
Generate a dataset with 0.1% prevalence:
1. Train a logistic regression model
2. Plot both ROC and PR curves
3. Calculate AUROC and AUPRC
4. Explain why they differ and which is more informative

### Exercise 3: Threshold Selection
For the sepsis model:
1. Find the threshold that achieves:
   - Exactly 95% sensitivity
   - Exactly 90% specificity
2. Calculate PPV at each threshold (assume 2% prevalence)
3. Which would you deploy if:
   - Missed sepsis is very costly
   - False alarms overwhelm ICU staff

### Exercise 4: Model Comparison
Train three models (Logistic Regression, Random Forest, XGBoost):
1. Plot all three ROC curves on the same plot
2. Identify which model is best at:
   - High sensitivity (> 90%)
   - Low FPR (< 10%)
   - Overall AUROC
3. Can the same model be best at all three? Why or why not?

---

**Next Notebook**: 5.3 Calibration and Clinical Utility (David's CVD Risk Journey)